In [10]:
!pip install pyodbc pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=AdventureWorks2025;"
    "Trusted_Connection=yes;"
)

query = """
SELECT
    sod.SalesOrderID,
    sod.SalesOrderDetailID,
    sod.ProductID,
    sod.OrderQty,
    sod.UnitPrice,
    sod.UnitPriceDiscount,
    sod.LineTotal,
    sod.ModifiedDate,
    p.Name AS ProductName,
    p.ProductNumber,
    p.Color,
    p.Class,
    p.Style
FROM Sales.SalesOrderDetail sod
LEFT JOIN Production.Product p
    ON sod.ProductID = p.ProductID
"""

df = pd.read_sql(query, conn)
df.head()
print(df.shape)

C:\Users\hisuk\AppData\Local\Temp\ipykernel_27068\2132472811.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


(121317, 13)


In [12]:
import numpy as np

name = df["ProductName"].fillna("").str.lower()

conditions = [
    # 🚴 Bike
    name.str.contains("bike|road-|mountain-|touring-"),

    # 👕 Apparel
    name.str.contains("shorts|jersey|vest|jacket|pants"),

    # 🧤 Accessories
    name.str.contains("gloves|socks|cap|bottle|rack|cleaner|wash"),

    # ⛑️ Protection
    name.str.contains("helmet"),

]

choices = [
    "Bike",
    "Apparel",
    "Accessories",
    "Protection",
]

df["ProductCategory"] = np.select(conditions, choices, default="Others")

In [13]:
conditions = [
    name.str.contains("mountain"),
    name.str.contains("road"),
    name.str.contains("touring"),
]

choices = [
    "Mountain Bike",
    "Road Bike",
    "Touring Bike",
]

df["BikeType"] = np.select(conditions, choices, default="Other")

In [14]:
df.to_csv("adventureworks_productgroup.csv", index=False)